In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "XRPUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/Users/maverick/Documents/Hackathon/quant_hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 285,120


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,volume_std_20,volume_z,taker_sell_base_asset_volume,taker_buy_ratio,imbalance,imbalance_5,imbalance_15,trend_strength,vol_regime_ratio,is_trending
0,2025-09-01 00:00:00+00:00,2.7757,2.7757,2.7723,2.7757,83483.8,2025-09-01 00:00:59.999999+00:00,2.315646e+05,1015,58143.3,...,NaN,NaN,25340.5,0.696462,0.392924,NaN,NaN,NaN,NaN,0
1,2025-09-01 00:01:00+00:00,2.7757,2.7772,2.7753,2.7772,84999.6,2025-09-01 00:01:59.999999+00:00,2.359481e+05,696,62509.2,...,NaN,NaN,22490.4,0.735406,0.470812,NaN,NaN,NaN,NaN,0
2,2025-09-01 00:02:00+00:00,2.7772,2.7774,2.7756,2.7767,36682.2,2025-09-01 00:02:59.999999+00:00,1.018428e+05,582,21315.5,...,NaN,NaN,15366.7,0.581086,0.162171,NaN,NaN,NaN,NaN,0
3,2025-09-01 00:03:00+00:00,2.7766,2.7770,2.7751,2.7751,28222.0,2025-09-01 00:03:59.999999+00:00,7.834349e+04,843,11713.8,...,NaN,NaN,16508.2,0.415059,-0.169882,NaN,NaN,NaN,NaN,0
4,2025-09-01 00:04:00+00:00,2.7751,2.7751,2.7605,2.7629,838170.8,2025-09-01 00:04:59.999999+00:00,2.318988e+06,4641,203901.1,...,NaN,NaN,634269.7,0.243269,-0.513462,0.068513,NaN,NaN,NaN,0


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 285,042
[info] optuna train rows: 182,426
[info] valid rows:        45,607
[info] test rows:         57,009


In [9]:
pruner = MedianPruner(n_warmup_steps=5, n_min_trials=10)
study = optuna.create_study(direction="maximize", pruner=pruner)

class EarlyStoppingCallback:
    def __init__(self, patience: int):
        self.patience = patience
        self.best_value = -float('inf')
        self.no_improvement_count = 0

    def __call__(self, study, trial):
        if study.best_value > self.best_value:
            self.best_value = study.best_value
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1

        if self.no_improvement_count >= self.patience:
            study.stop()

early_stopping = EarlyStoppingCallback(patience=10)

objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, callbacks=[early_stopping], show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-18 23:59:55,078] A new study created in memory with name: no-name-2fa3b83d-9e71-41db-bfaa-99bd2373a9d8


  0%|                                                                                                                  | 0/50 [00:00<?, ?it/s]

  0%|                                                                                                                  | 0/50 [00:03<?, ?it/s]

Best trial: 0. Best value: 0.0128086:   0%|                                                                            | 0/50 [00:03<?, ?it/s]

Best trial: 0. Best value: 0.0128086:   2%|█▎                                                                  | 1/50 [00:03<03:11,  3.90s/it]

...

Best trial: 17. Best value: 0.0231959:  54%|███████████████████████████████████▋                              | 27/50 [02:21<02:28,  6.47s/it]

Best trial: 17. Best value: 0.0231959:  54%|███████████████████████████████████▋                              | 27/50 [02:21<02:28,  6.47s/it]

Best trial: 17. Best value: 0.0231959:  56%|████████████████████████████████████▉                             | 28/50 [02:21<02:09,  5.87s/it]

Best trial: 17. Best value: 0.0231959:  56%|████████████████████████████████████▉                             | 28/50 [02:21<01:51,  5.07s/it]

[I 2026-03-19 00:02:16,972] Trial 27 finished with value: 0.010344422223250056 and parameters: {'n_estimators': 150, 'max_depth': 5, 'min_samples_split': 134, 'min_samples_leaf': 63, 'max_features': 'sqrt'}. Best is trial 17 with value: 0.0231959085385645.

[optuna] best trial
value: 0.023196
params:
  n_estimators: 200
  max_depth: 6
  min_samples_split: 159
  min_samples_leaf: 68
  max_features: sqrt


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final rf...


[training] done in 9.07s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.160522
Test IC:       -0.002359
Train Rank IC: 0.043729
Test Rank IC:  0.019084
Train RMSE:    0.002793
Test RMSE:     0.002363


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
vol_15              0.115550
vol_30              0.108533
range_5             0.100113
vol_5               0.077171
range_15            0.071664
mom_5               0.061256
mom_10              0.056198
mom_15              0.054648
dist_ma_15          0.047242
dist_ma_30          0.045372
mom_3               0.041917
dist_ma_5           0.041519
bar_range           0.036888
vol_regime_ratio    0.033952
range_ratio         0.029065
dist_ma_15_z        0.017599
imbalance_5         0.015243
imbalance_15        0.010694
volume_z            0.010022
volume_mom_5        0.008242
vol_ratio_5_30      0.007861
trend_strength      0.006082
is_trending         0.003169
dtype: float64


In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/XRPUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/XRPUSDT__h5_model.joblib
[saved] features -> models/rf/XRPUSDT__h5_feature_cols.json
[saved] feature importance -> models/rf/XRPUSDT__h5_feature_importance.csv
[saved] metadata -> models/rf/XRPUSDT__h5_meta.json
